In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 37.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import random
from faker import Faker
import json

# Configurar Faker e random
fake = Faker('pt_BR')
random.seed(42)

# ---------- USERS ----------
def gerar_usuario(id_usuario):
    nome = fake.first_name()
    email = fake.email()
    sexo = random.choice(['M', 'F'])
    idade = random.randint(18, 70)
    return {
        "id_usuario": id_usuario,
        "nome": nome,
        "email": email,
        "sexo": sexo,
        "idade": idade
    }

usuarios = [gerar_usuario(i+1) for i in range(50)]
df_usuarios = pd.DataFrame(usuarios)
df_usuarios.to_csv("users.csv", index=False)

# ---------- PRODUCTS ----------
categorias = ['Suplemento', 'Acessório Fitness', 'Comida Saudável']
produtos = []

for i in range(1, 21):
    produto = {
        "id_produto": i,
        "nome_produto": fake.word().capitalize() + " Pro",
        "preco": f"${round(random.uniform(20, 300), 2)}",
        "categoria": random.choice(categorias)
    }
    produtos.append(produto)

with open("products.json", "w", encoding="utf-8") as f:
    json.dump(produtos, f, ensure_ascii=False, indent=2)

# ---------- SALES ----------
def gerar_venda(id_venda):
    return {
        "id_venda": id_venda,
        "id_usuario": random.randint(1, 50),
        "id_produto": random.randint(1, 20),
        "quantidade": random.randint(1, 5),
        "data": fake.date_between(start_date='-1y', end_date='today').strftime('%Y-%m-%d')
    }

vendas = [gerar_venda(i+1) for i in range(100)]
df_vendas = pd.DataFrame(vendas)
df_vendas.to_csv("sales.csv", index=False)


In [3]:
categorias = ['Suplemento', 'Acessório Fitness', 'Comida Saudável']
produtos = []

for i in range(1, 21):
    produto = {
        "id_produto": i,
        "nome_produto": fake.word().capitalize() + " Pro",
        "preco": f"${round(random.uniform(20, 300), 2)}",
        "categoria": random.choice(categorias)
    }
    produtos.append(produto)

with open("products.json", "w", encoding="utf-8") as f:
    for produto in produtos:
        f.write(json.dumps(produto, ensure_ascii=False) + "\n")

In [4]:
from google.colab import files

files.download("sales.csv")
files.download("products.json")
files.download("users.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("Projeto PySpark") \
  .getOrCreate()

In [6]:
df_users = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("users.csv")

df_users.show(5)

+----------+---------+--------------------+----+-----+
|id_usuario|     nome|               email|sexo|idade|
+----------+---------+--------------------+----+-----+
|         1|     Igor|aparecidasarah@ex...|   M|   19|
|         2| Benjamim|mathiasaragao@exa...|   F|   33|
|         3|Guilherme|maria-clarajesus@...|   M|   26|
|         4|  Marcela|piresthales@examp...|   M|   61|
|         5|   Ágatha|maria-alice78@exa...|   M|   55|
+----------+---------+--------------------+----+-----+
only showing top 5 rows



In [7]:
df_sales = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("sales.csv")

df_sales.show(5)

+--------+----------+----------+----------+----------+
|id_venda|id_usuario|id_produto|quantidade|      data|
+--------+----------+----------+----------+----------+
|       1|        44|        18|         3|2025-05-14|
|       2|        50|        11|         1|2024-09-19|
|       3|        19|        14|         2|2025-03-06|
|       4|        30|         1|         3|2025-03-25|
|       5|        33|         6|         5|2024-10-08|
+--------+----------+----------+----------+----------+
only showing top 5 rows



In [8]:
df_products = spark.read.format("json") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("products.json")

In [9]:
df_products.show(5)

+-----------------+----------+-------------+-------+
|        categoria|id_produto| nome_produto|  preco|
+-----------------+----------+-------------+-------+
|  Comida Saudável|         1|      Est Pro| $64.58|
|       Suplemento|         2|      Qui Pro| $50.27|
|       Suplemento|         3|Assumenda Pro|$260.41|
|Acessório Fitness|         4|    Neque Pro| $75.88|
|       Suplemento|         5|    Rerum Pro| $117.9|
+-----------------+----------+-------------+-------+
only showing top 5 rows



## Perguntas a serem Respondidas

In [10]:
#1.Declare um novo dataframe que mostre o nome do produto e o valor final da compra.
#2. Declare um novo dataframe com o valor total gasto por cliente.
#3. Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o periodo.
#4. Declare um novo dataframe com os cinco produtos mais vendidos nos últimos seis meses (considerando período de dados disponível nos arquivos) contendo o nome do
#produto e a quantidade de produtos vendidos nesse periodo.
#5. Calcular a média de faturamento por cliente e o desvio padrão
#6. Classificar os clientes em três categorias silver. gold, platinum platinum: clientes que gastaram mais que a média de faturamento por cliente: gold: clientes que gastaram do menor desvio padrão até a média de faturamento por cliente; silver, clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;
#7. Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

##Vamos

In [11]:
#bora tratar os dados
#removendo valores nulos
df_users = df_users.dropna()
df_users = df_users.dropDuplicates()

df_products = df_products.dropna()
df_products = df_products.dropDuplicates()

df_sales = df_sales.dropna()
df_sales = df_sales.dropDuplicates()

In [12]:
#verificando schema e tipo
df_users.printSchema()

root
 |-- id_usuario: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- idade: integer (nullable = true)



In [13]:
#o preco ta como string, precisamos tratar
df_products.printSchema()

root
 |-- categoria: string (nullable = true)
 |-- id_produto: long (nullable = true)
 |-- nome_produto: string (nullable = true)
 |-- preco: string (nullable = true)



In [14]:
#ta como string e tem a string "$", preciso remover primeiro
df_products.show(3)

+-----------------+----------+------------+------+
|        categoria|id_produto|nome_produto| preco|
+-----------------+----------+------------+------+
|       Suplemento|        20|   Nihil Pro|$73.22|
|Acessório Fitness|         4|   Neque Pro|$75.88|
|  Comida Saudável|        15|   Autem Pro|$30.71|
+-----------------+----------+------------+------+
only showing top 3 rows



In [15]:
#vou usar regexp_replace para limpar antes de converter para numero
#importando o regexp replace
from pyspark.sql.functions import regexp_replace
#removendo o $ e mudando o type string para double
df_products = df_products.withColumn("preco", regexp_replace("preco", "[$]", "").cast("double"))
#conferindo se ta ok
df_products.printSchema()
df_products.show(3)

root
 |-- categoria: string (nullable = true)
 |-- id_produto: long (nullable = true)
 |-- nome_produto: string (nullable = true)
 |-- preco: double (nullable = true)

+-----------------+----------+------------+-----+
|        categoria|id_produto|nome_produto|preco|
+-----------------+----------+------------+-----+
|       Suplemento|        20|   Nihil Pro|73.22|
|Acessório Fitness|         4|   Neque Pro|75.88|
|  Comida Saudável|        15|   Autem Pro|30.71|
+-----------------+----------+------------+-----+
only showing top 3 rows



In [16]:
#agora bora pro sales
df_sales.printSchema()
#tudo ok com a data, se não iriamos converter para date
df_sales.show(3)

root
 |-- id_venda: integer (nullable = true)
 |-- id_usuario: integer (nullable = true)
 |-- id_produto: integer (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- data: date (nullable = true)

+--------+----------+----------+----------+----------+
|id_venda|id_usuario|id_produto|quantidade|      data|
+--------+----------+----------+----------+----------+
|      43|         1|         3|         2|2025-01-12|
|      50|        10|         7|         3|2024-08-15|
|      31|        37|        16|         2|2025-03-02|
+--------+----------+----------+----------+----------+
only showing top 3 rows



#AGORA VAMOS RESPONDER AS QUESTOES
#1.Declare um novo dataframe que mostre o nome do produto e o valor final da compra.


In [17]:
from pyspark.sql.functions import col, round

#fazendo junção entre sales e products
df_vendas_produtos = df_sales.join(
    df_products,
    on="id_produto",
    how="inner"
    )
#imprimindo pra ver se não deu ruim
#tudo Ok!
df_vendas_produtos.show(3)

+----------+--------+----------+----------+----------+-----------------+-------------+------+
|id_produto|id_venda|id_usuario|quantidade|      data|        categoria| nome_produto| preco|
+----------+--------+----------+----------+----------+-----------------+-------------+------+
|         3|      43|         1|         2|2025-01-12|       Suplemento|Assumenda Pro|260.41|
|         7|      50|        10|         3|2024-08-15|Acessório Fitness| Eligendi Pro| 74.08|
|        16|      31|        37|         2|2025-03-02|  Comida Saudável|     Sunt Pro|141.68|
+----------+--------+----------+----------+----------+-----------------+-------------+------+
only showing top 3 rows



In [18]:
#calcular o valor final (quantidade vezes preco)
df_valor_final = df_vendas_produtos.withColumn(
    "valor_final",
    round(col("quantidade") * col("preco"), 2)
)
#bora exibir somente o que importa na questao
df_resultado = df_valor_final.select("nome_produto", "valor_final")

df_resultado.show(5)

+-------------+-----------+
| nome_produto|valor_final|
+-------------+-----------+
|Assumenda Pro|     520.82|
| Eligendi Pro|     222.24|
|     Sunt Pro|     283.36|
|     Quis Pro|     131.91|
|Doloribus Pro|      480.4|
+-------------+-----------+
only showing top 5 rows



#2. Declare um novo dataframe com o valor total gasto por cliente.

In [19]:
from pyspark.sql.functions import sum
#agrupando para ver o valor total gasto
df_total_por_cliente = df_valor_final.groupBy("id_usuario") \
  .agg(round(sum("valor_final"), 2) \
  .alias("valor_total_gasto")
  )
#bora ver o resultado
df_total_por_cliente.show(3)

+----------+-----------------+
|id_usuario|valor_total_gasto|
+----------+-----------------+
|        31|           436.32|
|        34|           426.44|
|        26|           450.88|
+----------+-----------------+
only showing top 3 rows



#3. Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o periodo.

In [20]:
#juntar o total por cliente com o df users para obter o nome e email
df_melhores_clientes = df_total_por_cliente.join(
    df_users,
    on="id_usuario",
    how="inner"
)

#selecionando colunas relevantes
df_melhores_clientes = df_melhores_clientes.select(
    "nome", "email", "valor_total_gasto"
)

#vamos ordenar do maior para o menor e pegar os 5 primeiros
df_top5_clientes = df_melhores_clientes.orderBy(
    col("valor_total_gasto").desc()
).limit(5)

#bora ver o resultado, usando o truncate pra nao cortar os emails
df_top5_clientes.show(truncate=False)

+--------------+-------------------------+-----------------+
|nome          |email                    |valor_total_gasto|
+--------------+-------------------------+-----------------+
|Lucca         |enzo34@example.com       |2644.54          |
|Pedro Henrique|marcelanunes@example.net |2370.43          |
|Thomas        |isampaio@example.org     |2022.87          |
|Cauã          |luigisiqueira@example.org|1597.6           |
|Ryan          |xandrade@example.net     |1456.16          |
+--------------+-------------------------+-----------------+



#4. Declare um novo dataframe com os cinco produtos mais vendidos no ultimo mes (considerando período de dados disponível nos arquivos) contendo o nome do produto e a quantidade de produtos vendidos nesse periodo.

In [21]:
from pyspark.sql.functions import max, date_sub, lit

#como sabemos que a coluna data ja ta no formato date vamos direto ver qual a data mais recente
data_maxima = df_vendas_produtos.select(max("data")).first()[0]
data_corte = date_sub(lit(data_maxima), 30)

In [22]:
#filtrando ultimo mes
df_vendas_ultimo_mes = df_vendas_produtos.filter(
    col("data") >= data_corte
)

In [23]:
#agrupando por produto e somando a quantidade
df_top_produtos = df_vendas_ultimo_mes.groupBy("nome_produto") \
  .agg(sum("quantidade") \
  .alias("quantidade_vendida")) \
  .orderBy(col("quantidade_vendida") \
  .desc()) \
  .limit(5)

#visualizando
df_top_produtos.show(truncate=False)

+--------------+------------------+
|nome_produto  |quantidade_vendida|
+--------------+------------------+
|Animi Pro     |7                 |
|Reiciendis Pro|5                 |
|Inventore Pro |3                 |
|Doloribus Pro |2                 |
|Unde Pro      |1                 |
+--------------+------------------+



#5. Calcular a média de faturamento por cliente e o desvio padrão


In [24]:
from pyspark.sql.functions import mean, stddev

In [25]:
df_total_por_cliente.show()

+----------+-----------------+
|id_usuario|valor_total_gasto|
+----------+-----------------+
|        31|           436.32|
|        34|           426.44|
|        26|           450.88|
|        27|           294.03|
|        44|          2022.87|
|        12|          1302.05|
|        22|            322.9|
|         1|            753.6|
|        13|           758.96|
|         6|           351.68|
|        16|           488.55|
|         3|           757.34|
|        40|          1277.44|
|        20|            98.01|
|        48|           701.13|
|         5|           1176.8|
|        19|            140.0|
|        43|          1456.16|
|        15|            64.58|
|        37|           283.36|
+----------+-----------------+
only showing top 20 rows



In [26]:
#calculando estatisticas sobre o total gasto por cliente
estatisticas = df_total_por_cliente.agg(
    mean("valor_total_gasto").alias("media_faturamento"),
    stddev("valor_total_gasto").alias("desvio_padrao")
).first()

#separando os valores da Row
media_faturamento = estatisticas["media_faturamento"]
desvio_padrao = estatisticas["desvio_padrao"]

#exibindo os valores
print(f"Media faturamento é: , {media_faturamento:.2f}")
print(f"Desvio padrão é: {desvio_padrao:.2f}")

Media faturamento é: , 770.45
Desvio padrão é: 594.37


#6. Classificar os clientes em três categorias silver. gold, platinum platinum: clientes que gastaram mais que a média de faturamento por cliente: gold: clientes que gastaram do menor desvio padrão até a média de faturamento por cliente; silver, clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;


In [27]:
from pyspark.sql.functions import when, col

In [28]:
#definindo os limites
limite_silver = media_faturamento - desvio_padrao
limite_gold = media_faturamento

#classificando os clientes
df_categorias = df_total_por_cliente.withColumn(
    "categoria",
    when(col("valor_total_gasto") > limite_gold, "Platinum")
    .when((col("valor_total_gasto") > limite_silver)
    & (col("valor_total_gasto") <= limite_gold), "Gold")
    .otherwise("Silver")
)
df_categorias = df_categorias.withColumnRenamed("categoria", "categoria_cliente")

#exibindo o resultado
df_categorias.show(truncate=False)

+----------+-----------------+-----------------+
|id_usuario|valor_total_gasto|categoria_cliente|
+----------+-----------------+-----------------+
|31        |436.32           |Gold             |
|34        |426.44           |Gold             |
|26        |450.88           |Gold             |
|27        |294.03           |Gold             |
|44        |2022.87          |Platinum         |
|12        |1302.05          |Platinum         |
|22        |322.9            |Gold             |
|1         |753.6            |Gold             |
|13        |758.96           |Gold             |
|6         |351.68           |Gold             |
|16        |488.55           |Gold             |
|3         |757.34           |Gold             |
|40        |1277.44          |Platinum         |
|20        |98.01            |Silver           |
|48        |701.13           |Gold             |
|5         |1176.8           |Platinum         |
|19        |140.0            |Silver           |
|43        |1456.16 

#7. Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

In [29]:
from pyspark.sql.functions import sum as _sum, col, row_number, desc
from pyspark.sql.window import Window

df_vendas_categorias = df_valor_final.join(
    df_categorias.select("id_usuario", "categoria_cliente"), on="id_usuario", how="inner")

df_agrupado = df_vendas_categorias.groupBy("categoria_cliente", "nome_produto") \
  .agg({"quantidade":"sum"}) \
  .withColumnRenamed("sum(quantidade)", "total_vendido")

janela = Window.partitionBy("categoria_cliente").orderBy(desc("total_vendido"))

df_top3_produtos = df_agrupado.withColumn("rank", row_number().over(janela)) \
  .filter(col("rank") <= 3) \
  .drop("rank")

df_top3_produtos.show()

+-----------------+--------------+-------------+
|categoria_cliente|  nome_produto|total_vendido|
+-----------------+--------------+-------------+
|             Gold|  Adipisci Pro|           12|
|             Gold|     Animi Pro|            9|
|             Gold|       Est Pro|            8|
|         Platinum|Reiciendis Pro|           23|
|         Platinum|       Qui Pro|           19|
|         Platinum|     Harum Pro|           17|
|           Silver|     Harum Pro|            2|
|           Silver|      Unde Pro|            2|
|           Silver|     Nihil Pro|            1|
+-----------------+--------------+-------------+

